# ICT4403 Assignment 3 — English to Hindi Machine Translation
**Student:** Aryan Luke Monteiro | **Registration:** 230953462

Compare:
1. Custom basic encoder–decoder Transformer (trained from scratch)
2. `google/mt5-small` (fine-tuned)
3. `meta-llama/Llama-3.2-1B` (LoRA adaptation)
4. `google/gemma-2-2b` (LoRA adaptation)

Dataset: IIT Bombay English–Hindi Parallel Corpus — https://huggingface.co/datasets/cfilt/iitb-english-hindi

Run in Google Colab with a GPU. The notebook is a starter implementation. Confirm every cell runs in your selected Colab environment, and report only measured results. LLaMA/Gemma access may require accepting Hugging Face model terms.

In [ ]:
!pip -q install -U datasets transformers accelerate sentencepiece sacrebleu peft bitsandbytes scikit-learn pandas matplotlib

In [ ]:
import os, re, time, json, math, random, gc
import numpy as np, pandas as pd, torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
import sacrebleu
SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 1. Dataset inspection and preparation
Load the public IIT Bombay corpus, inspect the schema, then create reproducible train/validation/test splits.

In [ ]:
raw=load_dataset("cfilt/iitb-english-hindi")
print(raw)
for k in raw:
    print(k, raw[k].column_names, len(raw[k]))
print(raw["train"][0])

In [ ]:
def extract_pair(row):
    t=row.get("translation", {})
    return {"en":str(t.get("en","")).strip(), "hi":str(t.get("hi","")).strip()}
pairs=raw["train"].map(extract_pair, remove_columns=raw["train"].column_names)
pairs=pairs.filter(lambda r: bool(r["en"]) and bool(r["hi"]) and len(r["en"])<500 and len(r["hi"])<500)
pairs=pairs.shuffle(seed=SEED)
MAX_PAIRS=20000
pairs=pairs.select(range(min(MAX_PAIRS,len(pairs))))
df=pairs.to_pandas().drop_duplicates(["en","hi"])
train_df,tmp=train_test_split(df,test_size=.2,random_state=SEED)
valid_df,test_df=train_test_split(tmp,test_size=.5,random_state=SEED)
train_df=train_df.reset_index(drop=True); valid_df=valid_df.reset_index(drop=True); test_df=test_df.reset_index(drop=True)
print("Rows:",len(train_df),len(valid_df),len(test_df))

## 2. Shared evaluation

In [ ]:
def evaluate_predictions(preds, refs):
    return {"BLEU":sacrebleu.corpus_bleu(preds,[refs]).score,
            "chrF":sacrebleu.corpus_chrf(preds,[refs]).score}
def parameter_count(model):
    return sum(p.numel() for p in model.parameters())
test_sample=test_df.head(min(200,len(test_df))).copy()
results=[]
samples={}

## 3. Basic Transformer — encoder–decoder, trained from scratch
This is a compact baseline. It uses a shared word-level vocabulary and PyTorch's Transformer encoder–decoder. Increase epochs or dataset size only if GPU time permits.

In [ ]:
from collections import Counter
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
PAD,BOS,EOS,UNK="<pad>","<bos>","<eos>","<unk>"
def tok(s): return re.findall(r"\\w+|[^\\w\\s]",str(s).lower(),flags=re.UNICODE)
cnt=Counter()
for s in train_df.en: cnt.update(tok(s))
for s in train_df.hi: cnt.update(tok(s))
stoi={PAD:0,BOS:1,EOS:2,UNK:3}
for w,_ in cnt.most_common(12000):
    if w not in stoi: stoi[w]=len(stoi)
itos={v:k for k,v in stoi.items()}; V=len(stoi); MAXLEN=64
def enc(s): return [1]+[stoi.get(w,3) for w in tok(s)[:MAXLEN-2]]+[2]
class Pairs(TorchDataset):
    def __init__(self,frame): self.f=frame
    def __len__(self): return len(self.f)
    def __getitem__(self,i):
        r=self.f.iloc[i]; return enc(r.en),enc(r.hi)
def collate(batch):
    a,b=zip(*batch)
    pad=lambda xs: torch.tensor([x+[0]*(max(map(len,xs))-len(x)) for x in xs])
    return pad(a),pad(b)
train_loader=DataLoader(Pairs(train_df),batch_size=48,shuffle=True,collate_fn=collate)
valid_loader=DataLoader(Pairs(valid_df),batch_size=48,collate_fn=collate)
class BasicTransformer(nn.Module):
    def __init__(self):
        super().__init__(); d=192; self.d=d
        self.se=nn.Embedding(V,d,padding_idx=0); self.te=nn.Embedding(V,d,padding_idx=0)
        self.pos=nn.Embedding(MAXLEN,d)
        self.tr=nn.Transformer(d_model=d,nhead=4,num_encoder_layers=2,num_decoder_layers=2,
                               dim_feedforward=384,batch_first=True,dropout=.1)
        self.out=nn.Linear(d,V)
    def forward(self,src,tgt):
        ps=torch.arange(src.size(1),device=src.device)[None,:]
        pt=torch.arange(tgt.size(1),device=tgt.device)[None,:]
        s=self.se(src)*math.sqrt(self.d)+self.pos(ps); t=self.te(tgt)*math.sqrt(self.d)+self.pos(pt)
        mask=torch.triu(torch.ones(tgt.size(1),tgt.size(1),device=tgt.device,dtype=torch.bool),1)
        z=self.tr(s,t,tgt_mask=mask,src_key_padding_mask=src.eq(0),
                  tgt_key_padding_mask=tgt.eq(0),memory_key_padding_mask=src.eq(0))
        return self.out(z)
model=BasicTransformer().to(device)
opt=torch.optim.AdamW(model.parameters(),lr=3e-4)
loss_fn=nn.CrossEntropyLoss(ignore_index=0)
def epoch(loader,training):
    model.train(training); ls=[]
    for src,tgt in loader:
        src,tgt=src.to(device),tgt.to(device)
        with torch.set_grad_enabled(training):
            logits=model(src,tgt[:,:-1]); loss=loss_fn(logits.reshape(-1,V),tgt[:,1:].reshape(-1))
            if training:
                opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1); opt.step()
        ls.append(loss.item())
    return float(np.mean(ls))
t0=time.time()
for e in range(3):
    print("Epoch",e+1,"train",epoch(train_loader,True),"valid",epoch(valid_loader,False))
basic_train_time=time.time()-t0
print("Training seconds",basic_train_time,"parameters",parameter_count(model))

In [ ]:
@torch.no_grad()
def basic_translate(text):
    model.eval(); src=torch.tensor([enc(text)],device=device); out=[BOS]
    for _ in range(60):
        logits=model(src,torch.tensor([[stoi[x] for x in out]],device=device))[:,-1]
        nxt=int(logits.argmax(-1))
        if nxt==2: break
        out.append(itos.get(nxt,UNK))
    return " ".join(out[1:])
t0=time.time(); basic_preds=[basic_translate(x) for x in test_sample.en]
basic_infer=time.time()-t0
basic_scores=evaluate_predictions(basic_preds,test_sample.hi.tolist())
results.append({"Model":"Basic Transformer","Train seconds":basic_train_time,"Inference seconds":basic_infer,
                "Parameters":parameter_count(model),**basic_scores})
samples["Basic Transformer"]=list(zip(test_sample.en.tolist()[:10],basic_preds[:10],test_sample.hi.tolist()[:10]))
print(basic_scores)

## 4. mT5-small — fine-tuning

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
mt5_name="google/mt5-small"
mt5_tok=AutoTokenizer.from_pretrained(mt5_name)
mt5=AutoModelForSeq2SeqLM.from_pretrained(mt5_name).to(device)
def prep(batch):
    x=["translate English to Hindi: "+s for s in batch["en"]]
    a=mt5_tok(x,max_length=128,truncation=True)
    a["labels"]=mt5_tok(text_target=batch["hi"],max_length=128,truncation=True)["input_ids"]
    return a
tr=Dataset.from_pandas(train_df[["en","hi"]]).map(prep,batched=True,remove_columns=["en","hi"])
va=Dataset.from_pandas(valid_df[["en","hi"]]).map(prep,batched=True,remove_columns=["en","hi"])
args=Seq2SeqTrainingArguments(output_dir="mt5_en_hi",num_train_epochs=1,learning_rate=2e-5,
    per_device_train_batch_size=4,per_device_eval_batch_size=4,gradient_accumulation_steps=4,
    evaluation_strategy="epoch",save_strategy="no",fp16=torch.cuda.is_available(),report_to="none")
trainer=Seq2SeqTrainer(model=mt5,args=args,train_dataset=tr,eval_dataset=va,
    data_collator=DataCollatorForSeq2Seq(mt5_tok,model=mt5),tokenizer=mt5_tok)
t0=time.time(); trainer.train(); mt5_train_time=time.time()-t0
def seq_generate(m,tok, texts, prefix=""):
    outs=[]
    for i in range(0,len(texts),8):
        batch=[prefix+x for x in texts[i:i+8]]
        x=tok(batch,return_tensors="pt",padding=True,truncation=True,max_length=128).to(device)
        with torch.no_grad(): y=m.generate(**x,max_new_tokens=100,num_beams=4)
        outs += tok.batch_decode(y,skip_special_tokens=True)
    return outs
t0=time.time(); mt5_preds=seq_generate(mt5,mt5_tok,test_sample.en.tolist(),"translate English to Hindi: ")
mt5_infer=time.time()-t0; sc=evaluate_predictions(mt5_preds,test_sample.hi.tolist())
results.append({"Model":"mT5-small","Train seconds":mt5_train_time,"Inference seconds":mt5_infer,
                "Parameters":parameter_count(mt5),**sc})
samples["mT5-small"]=list(zip(test_sample.en.tolist()[:10],mt5_preds[:10],test_sample.hi.tolist()[:10]))
print(sc)

## 5. LLaMA and Gemma — decoder-only LoRA adaptation
Model access may be gated. Accept the terms on each Hugging Face model page, then log in with a read token. Run each model in a separate Colab runtime if memory is tight. The two model runs are opt-in below.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import DataCollatorForLanguageModeling
def load_decoder(model_id):
    q=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,bnb_4bit_use_double_quant=True)
    tokz=AutoTokenizer.from_pretrained(model_id)
    if tokz.pad_token is None: tokz.pad_token=tokz.eos_token
    m=AutoModelForCausalLM.from_pretrained(model_id,quantization_config=q,device_map="auto")
    m=prepare_model_for_kbit_training(m)
    m=get_peft_model(m,LoraConfig(r=8,lora_alpha=16,lora_dropout=.05,
                                  target_modules="all-linear",task_type="CAUSAL_LM"))
    return m,tokz
def decoder_dataset(frame,tokz,limit=10000):
    texts=[]
    for r in frame.head(limit).itertuples():
        texts.append(f"Translate English to Hindi. Return only the Hindi translation.\\nEnglish: {r.en}\\nHindi: {r.hi}{tokz.eos_token}")
    d=Dataset.from_dict({"text":texts})
    return d.map(lambda b:tokz(b["text"],max_length=256,truncation=True),batched=True,remove_columns=["text"])
def run_decoder(model_id,label):
    m,t=load_decoder(model_id)
    d=decoder_dataset(train_df,t)
    a=TrainingArguments(output_dir=label.replace(" ","_"),num_train_epochs=1,learning_rate=2e-4,
        per_device_train_batch_size=1,gradient_accumulation_steps=8,logging_steps=50,
        save_strategy="no",fp16=torch.cuda.is_available(),report_to="none",
        optim="paged_adamw_8bit")
    trn=Trainer(model=m,args=a,train_dataset=d,
        data_collator=DataCollatorForLanguageModeling(t,mlm=False))
    start=time.time(); trn.train(); elapsed=time.time()-start
    return m,t,elapsed
def decoder_translate(m,t,text):
    p=f"Translate English to Hindi. Return only the Hindi translation.\\nEnglish: {text}\\nHindi:"
    x=t(p,return_tensors="pt").to(m.device)
    with torch.no_grad():
        y=m.generate(**x,max_new_tokens=100,do_sample=False,eos_token_id=t.eos_token_id,
                      pad_token_id=t.eos_token_id)
    return t.decode(y[0][x.input_ids.shape[1]:],skip_special_tokens=True).strip()

In [ ]:
# LLaMA: uncomment after confirming Hugging Face access.
# mid="meta-llama/Llama-3.2-1B"
# m,t,train_s=run_decoder(mid,"llama_lora")
# start=time.time(); pred=[decoder_translate(m,t,x) for x in test_sample.en.tolist()]; infer_s=time.time()-start
# sc=evaluate_predictions(pred,test_sample.hi.tolist())
# results.append({"Model":"LLaMA 3.2 1B","Train seconds":train_s,"Inference seconds":infer_s,
#                 "Parameters":parameter_count(m),**sc})
# samples["LLaMA 3.2 1B"]=list(zip(test_sample.en.tolist()[:10],pred[:10],test_sample.hi.tolist()[:10]))
# m.save_pretrained("llama_lora_adapter"); del m; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Gemma: ideally run in a fresh Colab GPU runtime after LLaMA.
# mid="google/gemma-2-2b"
# m,t,train_s=run_decoder(mid,"gemma_lora")
# start=time.time(); pred=[decoder_translate(m,t,x) for x in test_sample.en.tolist()]; infer_s=time.time()-start
# sc=evaluate_predictions(pred,test_sample.hi.tolist())
# results.append({"Model":"Gemma 2 2B","Train seconds":train_s,"Inference seconds":infer_s,
#                 "Parameters":parameter_count(m),**sc})
# samples["Gemma 2 2B"]=list(zip(test_sample.en.tolist()[:10],pred[:10],test_sample.hi.tolist()[:10]))
# m.save_pretrained("gemma_lora_adapter")

## 6. Save results and compare
Only claim a four-model comparison after all four have actually run.

In [ ]:
os.makedirs("results",exist_ok=True)
results_df=pd.DataFrame(results)
display(results_df)
results_df.to_csv("results/model_comparison.csv",index=False)
with open("results/generated_samples.json","w",encoding="utf-8") as f:
    json.dump(samples,f,ensure_ascii=False,indent=2)
if len(results_df):
    ax=results_df.set_index("Model")[["BLEU","chrF"]].plot(kind="bar",figsize=(9,5))
    ax.set_ylabel("Corpus score"); ax.set_title("English–Hindi translation quality")
    ax.figure.tight_layout(); ax.figure.savefig("results/quality_comparison.png",dpi=160)

## 7. Responsible AI analysis
Inspect actual translations for:
- bias or stereotyping;
- hallucinated additions or omitted meaning;
- cultural misrepresentation, idioms, honorifics and named entities;
- unsafe or inappropriate translation of sensitive content.

Discuss mitigations: human review for high-impact uses, domain-specific testing, bias audits, careful dataset documentation, uncertainty/abstention, named-entity checks, and disclosure that outputs are machine-generated. A small test set cannot establish that a system is unbiased or safe.

## 8. Reporting checklist
Record GPU type, exact dataset split sizes, model revisions, epochs, batch sizes, training/fine-tuning time, inference time, parameter count, BLEU and chrF. Explain that pretrained models, tokenizers, sizes and adaptation methods differ. Do not fabricate missing metrics.